# 8.2 Kodierung und Skalierung

Bisher haben wir zur Prognose des Verkaufspreises eines Autos nur numerische
Merkmale wie den Kilometerstand genutzt. Viele Merkmale liegen aber als Text vor,
zum Beispiel die Farbe oder der Kraftstoff. Andere numerische Merkmale liegen in
sehr unterschiedlichen Größenordnungen. Damit ein ML-Modell auch diese Merkmale
nutzen kann, bereiten wir die Daten in zwei Schritten vor: mit der Kodierung
von kategorialen Daten und der Skalierung von numerischen Daten.

## Geordnete kategoriale Daten kodieren

Bei den Beispielen zur linearen Regression haben wir zur Prognose des
Verkaufspreises nur numerische Merkmale genutzt, zum Beispiel den Kilometerstand.
Es gibt aber weitere Merkmale, die den Preis beeinflussen, etwa den Kraftstoff
(Diesel oder Benzin) oder den Zustand des Autos. Diese Merkmale liegen als Text
vor, in Pandas erkennbar am Datentyp `object`. Ein ML-Modell kann mit Text nicht
rechnen. Wir müssen die Kategorien vorher in Zahlen übersetzen. Diesen Schritt
nennen wir **Kodierung**. Wie wir das tun, hängt davon ab, ob die Kategorien
eine natürliche Reihenfolge haben. In diesem Abschnitt behandeln wir geordnete
Kategorien.

Wir laden einen Datensatz mit Verkaufsdaten der Plattform
[Autoscout24.de](https://www.autoscout24.de). Mit der Methode `.info()` lassen
wir uns anzeigen, welchen Datentyp die Merkmale haben.

In [ ]:
import pandas as pd

daten = pd.read_csv('autoscout24_kodierung.csv')
daten.info()

Wir sehen

- 8 Merkmale mit Datentyp `object`: Marke, Modell, Farbe, Erstzulassung,
  Getriebe, Kraftstoff, Bemerkungen und Zustand,
- 4 Merkmale mit Datentyp `int64`: Jahr, Preis (Euro), Leistung (PS) und
  Leistung (kW),
- 2 Merkmale mit Datentyp `float64`: Verbrauch (l/100 km) und Kilometerstand
  (km).

Die acht Merkmale vom Typ `object` sind Text und müssen kodiert werden. Wir
beginnen mit dem Merkmal »Zustand«. Mit der Methode `.unique()` sehen wir, welche
Kategorien es gibt.

In [ ]:
daten['Zustand'].unique()

Es gibt vier Kategorien. Sie haben eine natürliche Reihenfolge, denn ein Neuwagen
ist mehr wert als ein Jahreswagen, dieser mehr als ein junger Gebrauchtwagen und
dieser mehr als ein Gebrauchtwagen. Geordnete Kategorien nennen wir **ordinale
Daten**. Beim Kodieren behalten wir die Reihenfolge bei:

- Gebrauchtwagen wird zu 0,
- junger Gebrauchtwagen wird zu 1,
- Jahreswagen wird zu 2,
- Neuwagen wird zu 3.

Diese Zuordnung schreiben wir in ein **Dictionary**. Ein Dictionary ordnet jedem
Eintrag einen Wert zu, hier also jeder Kategorie eine Zahl. Wir schreiben die
Zahlen zunächst als Text `'0'` bis `'3'`. Den Grund erklären wir gleich.

In [ ]:
zustand_kodierung = {
    'Gebrauchtwagen': '0',
    'junger Gebrauchtwagen': '1',
    'Jahreswagen': '2',
    'Neuwagen': '3',
}

Mit der Methode `.replace()` ersetzen wir jede Kategorie durch den Wert aus dem
Dictionary. Damit `.replace()` zuverlässig funktioniert, ersetzen wir Text durch
Text. Die Umwandlung in echte Zahlen machen wir danach in einem eigenen Schritt
mit der Methode `.astype()`.

In [ ]:
daten['Zustand'] = # TODO: ???  (Kategorien in der Spalte Zustand mit dem Dictionary ersetzen)
daten['Zustand'] = daten['Zustand'].astype('int')

# Kontrolle
daten['Zustand'].unique()

Aus den vier Kategorien sind die Zahlen 0 bis 3 geworden. Die Reihenfolge der
Zustände bleibt dabei erhalten.

Manche Merkmale haben nur zwei Kategorien, zum Beispiel »Getriebe« mit den
Werten Automatik und Schaltgetriebe. Hier gibt es keine natürliche Reihenfolge.
Bei genau zwei Kategorien spielt das aber keine Rolle. Die Zuordnung muss aber
eindeutig und überall gleich bleiben. Wir vergeben einfach 0 und 1 und gehen
dabei genauso vor wie beim Zustand.

## Ungeordnete kategoriale Daten kodieren

Bei vielen Merkmalen gibt es keine natürliche Reihenfolge. Ein Beispiel ist die
Farbe eines Autos.

In [ ]:
daten['Farbe'].unique()

Der Datensatz enthält 14 verschiedene Farben. Merkmale ohne natürliche
Reihenfolge nennen wir **nominale Daten**. Wir könnten den Farben die Zahlen 0
bis 13 geben. Damit würden wir aber eine Reihenfolge behaupten, die es nicht
gibt. Grau wäre kleiner als Grün, und der Abstand zwischen Grau und Grün wäre
genauso groß wie der zwischen zwei anderen Farben. Das führt ein ML-Modell in die
Irre.

Stattdessen verwenden wir die **One-Hot-Kodierung**. Wir ersetzen die eine Spalte
»Farbe« durch 14 neue Spalten, für jede Farbe eine. In der Zeile eines grauen
Autos steht in der Spalte für Grau eine 1 und in den 13 anderen Farbspalten eine
0. So ist jede Farbe als Zahl dargestellt, ohne dass eine Reihenfolge entsteht.

Pandas übernimmt diese Umwandlung mit der Methode `.get_dummies()`. Wir schauen
uns zuerst an, was sie mit der Spalte »Farbe« macht. Mit dem Argument `dtype=int`
legen wir fest, dass die neuen Spalten Nullen und Einsen enthalten.

In [ ]:
pd.get_dummies(daten['Farbe'], dtype=int)

Jede Zeile hat genau eine 1, nämlich in der Spalte der tatsächlichen Farbe. Damit
`.get_dummies()` diese Umwandlung direkt im ganzen Datensatz vornimmt, übergeben
wir mit `data` den kompletten Datensatz und mit `columns` eine Liste der Spalten,
die One-Hot-kodiert werden sollen.

In [ ]:
daten = # TODO: ???  (Spalte Farbe im ganzen Datensatz One-Hot-kodieren, neue Spalten mit Nullen und Einsen)
daten.head()

Die Spalte »Farbe« ist verschwunden. An ihrer Stelle stehen die neuen Spalten
`Farbe_grau`, `Farbe_rot` und so weiter. Der Name jeder neuen Spalte setzt sich
aus dem alten Spaltennamen und der Kategorie zusammen.

## Numerische Daten skalieren

Jetzt betrachten wir die numerischen Merkmale. Wir laden den Datensatz neu und
behalten nur die Spalten mit Zahlen.

In [ ]:
daten = pd.read_csv('autoscout24_kodierung.csv')
daten = daten.drop(columns=['Marke', 'Modell', 'Farbe', 'Erstzulassung',
                            'Getriebe', 'Kraftstoff', 'Bemerkungen', 'Zustand'])
daten.info()

Ein Blick auf die statistischen Kennzahlen zeigt, dass die Merkmale in sehr
verschiedenen Zahlenbereichen liegen.

In [ ]:
daten.describe()

Der Verbrauch liegt zwischen 3.5 und 14.9 Litern pro 100 Kilometer. Der
Kilometerstand geht dagegen bis über 400 000 km. Im Boxplot sehen wir deshalb fast
nur noch den Kilometerstand. Alle anderen Merkmale werden zu einem Strich am
unteren Rand gequetscht.

In [ ]:
import plotly.express as px

fig = px.box(daten)
fig.show()

Solche Größenunterschiede stören viele ML-Modelle beim Training. Ein Merkmal mit
großen Zahlen bekommt dann mehr Gewicht als ein Merkmal mit kleinen Zahlen,
obwohl das inhaltlich nicht gerechtfertigt ist. Beim **Skalieren** bringen wir
alle Merkmale in eine vergleichbare Größenordnung. Es gibt zwei gebräuchliche
Verfahren: die **Normierung** und die **Standardisierung**.

### Normierung

Bei der Normierung legen wir fest, dass alle Werte eines Merkmals in einem festen
Intervall liegen. Meist wählen wir das Intervall von 0 bis 1. Der kleinste Wert
eines Merkmals wird zu 0, der größte zu 1, alle anderen liegen dazwischen. Für
den Verbrauch heißt das: aus 3.5 wird 0, aus 14.9 wird 1.

Wir nutzen dafür die Klasse `MinMaxScaler` aus Scikit-Learn. Zuerst importieren
wir sie und erzeugen ein Objekt.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

normierung = MinMaxScaler()

Mit der Methode `.fit()` bestimmt der `MinMaxScaler` das Minimum und das Maximum
jeder Spalte.

In [ ]:
# TODO: ???  (Minimum und Maximum jeder Spalte bestimmen)

Mit der Methode `.transform()` rechnen wir die Werte um.

In [ ]:
daten_normiert = # TODO: ???  (Werte umrechnen)
print(daten_normiert)

Alle Werte liegen jetzt zwischen 0 und 1. Auffällig ist, dass wir kein
Pandas-DataFrame mehr haben, sondern ein NumPy-Array ohne Spaltennamen.
Scikit-Learn gibt skalierte Daten immer als NumPy-Array zurück, denn genau diese
Form braucht der ML-Algorithmus später.

Für einen Boxplot brauchen wir aber Spaltennamen. Wir bauen deshalb kurz ein
DataFrame aus dem Array und den ursprünglichen Spaltennamen.

In [ ]:
daten_normiert = pd.DataFrame(daten_normiert, columns=daten.columns)

fig = px.box(daten_normiert)
fig.show()

Jetzt liegen alle Merkmale im selben Bereich und sind im Boxplot gleich gut zu
erkennen.

### Standardisierung

Die Standardisierung bringt die Werte nicht in ein festes Intervall. Stattdessen
verschiebt sie jedes Merkmal so, dass sein Mittelwert 0 und seine
Standardabweichung 1 beträgt. Werte über dem Mittelwert werden positiv, Werte
darunter negativ. Diese Form passt gut zu Merkmalen, die annähernd normalverteilt
sind.

Das Vorgehen ist dasselbe wie bei der Normierung. Wir tauschen nur den Scaler aus
und verwenden die Klasse `StandardScaler`. Die Methode `.fit()` bestimmt hier den
Mittelwert und die Standardabweichung jeder Spalte.

In [ ]:
from sklearn.preprocessing import StandardScaler

standardisierung = StandardScaler()
standardisierung.fit(daten)

daten_standardisiert = standardisierung.transform(daten)
print(daten_standardisiert)

Auch hier ist das Ergebnis ein NumPy-Array. Zur Kontrolle bauen wir wieder ein
DataFrame und schauen uns die Kennzahlen an. Der Mittelwert jeder Spalte liegt
nun bei 0 und die Standardabweichung bei ungefähr 1.

In [ ]:
daten_standardisiert = pd.DataFrame(daten_standardisiert, columns=daten.columns)
daten_standardisiert.describe()

Der Boxplot zeigt dasselbe Bild wie die Kennzahlen.

In [ ]:
fig = px.box(daten_standardisiert)
fig.show()

Alle Merkmale liegen jetzt in einer vergleichbaren Größenordnung um die 0 herum.
Anders als bei der Normierung sind die Werte aber nicht auf ein festes Intervall
begrenzt. Ausreißer wie die teuersten Autos bleiben deshalb sichtbar.

Welches der beiden Verfahren besser passt, hängt vom ML-Modell und von den Daten
ab. Später trennen wir die Daten außerdem in Trainings- und Testdaten. Dann
passen wir den Scaler nur an die Trainingsdaten an, damit keine Information aus
den Testdaten in die Skalierung einfließt. Mehr dazu im nächsten Kapitel.

## Zusammenfassung und Ausblick

Kategoriale Merkmale müssen wir kodieren, bevor ein ML-Modell sie nutzen kann.
Geordnete Kategorien (ordinale Daten) kodieren wir über ein Dictionary und die
Methode `.replace()` als Zahlen und behalten dabei die Reihenfolge bei. Bei genau
zwei Kategorien spielt die Reihenfolge keine Rolle. Ungeordnete Kategorien
(nominale Daten) kodieren wir mit der One-Hot-Kodierung über die Methode
`.get_dummies()`.

Numerische Merkmale bringen wir durch Skalieren in eine vergleichbare
Größenordnung, entweder durch Normierung mit dem `MinMaxScaler` oder durch
Standardisierung mit dem `StandardScaler`. Für die lineare Regression und
Entscheidungsbäume ist das nicht nötig. Spätere Modelle wie neuronale Netze
profitieren in der Regel stark von skalierten Eingaben.

Im nächsten Kapitel trennen wir die Daten in Trainings- und Testdaten. Damit
können wir prüfen, wie gut ein Modell mit neuen, ungesehenen Daten zurechtkommt.

## Mini-Übungen

### Mini-Übung 1

Übertragen Sie das Vorgehen auf einen Datensatz aus der Fertigung. Die Datei
`3ddruck_kodierung.csv` enthält 200 3D-Druckaufträge mit jeweils 15 Merkmalen.

1. Lesen Sie die Datei ein. Die Spalte `Nummer` soll als Zeilenindex dienen.
   Verschaffen Sie sich einen Überblick über die Merkmale und ihre Datentypen.
2. Das Merkmal `Oberflaechenguete` ist geordnet: grob ist schlechter als mittel,
   mittel schlechter als fein. Kodieren Sie es als ganze Zahlen 0, 1 und 2 und
   behalten Sie dabei die Reihenfolge bei.
3. Das Merkmal `Erfolgreich` hat nur zwei Kategorien: ja und nein. Kodieren Sie
   es als 0 und 1.
4. Prüfen Sie bei beiden Merkmalen, ob nur noch Zahlen enthalten sind.

In [ ]:
# Code-Zelle

### Mini-Übung 2

Arbeiten Sie mit dem Datensatz aus der vorherigen Mini-Übung weiter.

1. Die Merkmale `Material`, `Farbe` und `Infill-Muster` haben keine natürliche
   Reihenfolge. Kodieren Sie alle drei Merkmale in einem Schritt mit der
   One-Hot-Kodierung. Die neuen Spalten sollen Nullen und Einsen enthalten.
2. Sehen Sie sich die ersten Zeilen des Ergebnisses an. Wie heißen die neuen
   Spalten, und wie viele Spalten hat der Datensatz jetzt?

In [ ]:
# Code-Zelle

### Mini-Übung 3

Lesen Sie `3ddruck_kodierung.csv` erneut ein. Die Spalte `Nummer` soll als
Zeilenindex dienen. Behalten Sie nur die numerischen Merkmale.

1. Verschaffen Sie sich über die statistischen Kennzahlen einen Überblick.
   Welches Merkmal hat den größten Zahlenbereich, welches den kleinsten?
2. Normieren Sie alle numerischen Merkmale auf das Intervall von 0 bis 1.
3. Standardisieren Sie dieselben Merkmale stattdessen auf Mittelwert 0 und
   Standardabweichung 1.
4. Prüfen Sie jeweils über die statistischen Kennzahlen oder einen Boxplot, ob
   die Skalierung geklappt hat.

In [ ]:
# Code-Zelle